In [1]:
import h5py
import numpy as np
import meshio

import matplotlib.pyplot as plt

from MeshManager import MeshManager


#Set fonts
from matplotlib import rc, rcParams, cm
rc('font', **{'family': 'sans-serif', 'sans-serif': ['Arial']})

In [2]:
# Plotting function
def PlotLine(y = 2*56.25/25.4, x = 1.5*56.25/25.4, dpi=100):
    fig, ax = plt.subplots(figsize=(y, x), dpi=dpi, tight_layout=True)
    return ax

In [ ]:
# Element name
elementName = "quad"  		# meshio compatible element name
mesh = MeshManager("../SmoothAxi.msh", elementName)

In [7]:
y1 = mesh.getNodesByGroup("Upper_gauge")
y2 = mesh.getNodesByGroup("Lower_gauge")

dofyt = int(2*y1[0]+1)
dofyb = int(2*y2[0]+1)

nSteps = 18
topY_DOFs = 2*mesh.getNodesByGroup("Top")+1
force_y = np.zeros((nSteps))
disp_y = np.zeros((nSteps))

In [8]:
with h5py.File("SRB_Air.mech.out.hdf5", "a") as fh5:
    
    for iStep in range(nSteps):
        
        force_y[iStep] = fh5["Force/Step_"+str(iStep)][()][topY_DOFs].sum()
        yt = fh5["Disp/Step_"+str(iStep)][()][dofyt]
        yb = fh5["Disp/Step_"+str(iStep)][()][dofyb]
        disp_y[iStep] = yt - yb

In [ ]:
ax = PlotLine()

ax.plot(disp_y/0.03, force_y*1e-6/(np.pi*(0.003)**2), "g", lw=2, label="FEM")

ax.set_xlabel("Eng. Strain $\\Delta L/L_0$")
ax.set_ylabel("Eng. Stress $\\sigma_{yy}$ (MPa)")

ax.legend(frameon=False)